In [ ]:
from google.cloud import bigquery
import warnings
import pandas as pd
from datasets import Dataset
from utils.hub_card import push_dataset_card
warnings.filterwarnings(action='ignore', message="Your application has authenticated using end user")

client = bigquery.Client(project="ads-599-final")

In [2]:
# Triage information from ed.triage — static, one row per ED visit.
# Chief complaint, ESI acuity level, and initial vitals recorded at triage.
# One-to-one with edstays, so row count should match cohort size.
# Used as baseline/static state features (acuity, chiefcomplaint) and initial vital signs.

query_triage = """
WITH cohort_subjects AS (
  SELECT ed_stay_id, subject_id
  FROM `ads-599-final.rl_project.cohort_base`
)
SELECT
  t.stay_id   AS ed_stay_id,
  t.subject_id,
  t.temperature,
  t.heartrate,
  t.resprate,
  t.o2sat,
  t.sbp,
  t.dbp,
  t.pain,
  t.acuity,
  t.chiefcomplaint
FROM `physionet-data.mimiciv_ed.triage` t
INNER JOIN cohort_subjects cs ON t.stay_id = cs.ed_stay_id
ORDER BY t.stay_id
"""

print("Running triage query...")
df_triage = client.query(query_triage).to_dataframe()
print(f"Shape: {df_triage.shape}")
print(f"Triage rows vs expected cohort size: {len(df_triage):,}  (should be ~399,573)")
df_triage.head()

Running triage query...
Shape: (399573, 11)
Triage rows vs expected cohort size: 399,573  (should be ~399,573)


,ed_stay_id,subject_id,temperature,heartrate,resprate,o2sat,sbp,dbp,pain,acuity,chiefcomplaint
0,30000012,11714491,98.800000000,96.000000000,18.000000000,93.000000000,160.000000000,54.000000000,0,2.000000000,CHANGE IN MENTAL STATUS
1,30000038,13821532,97.100000000,54.000000000,18.000000000,95.000000000,143.000000000,73.000000000,0,3.000000000,Cough
2,30000039,13340997,98.600000000,85.000000000,16.000000000,98.000000000,189.000000000,96.000000000,0,3.000000000,s/p Fall
3,30000055,19848164,99.400000000,85.000000000,16.000000000,100.000000000,None,None,0,3.000000000,L Ear pain
4,30000094,19862552,98.100000000,60.000000000,18.000000000,94.000000000,120.000000000,95.000000000,2,2.000000000,N


In [ ]:
DESCRIPTION = (
    "ED triage records from ed.triage. One row per ED visit. Includes ESI acuity level, "
    "chief complaint, and initial vital signs at triage. Includes back-to-back ED visits "
    "that are handled when merging with the cohort."
)

ds = Dataset.from_pandas(df_triage, split='triage')
ds.push_to_hub("ADS599-Capstone/raw_data", config_name="triage", data_dir="triage")
push_dataset_card("ADS599-Capstone/raw_data", config_name="triage", split="triage", data_dir="triage", description=DESCRIPTION)
print("Pushed triage to HuggingFace Hub.")

In [ ]:
from datasets import load_dataset

# Load the processed cohort (post AGAINST ADVICE removal, consecutive visit collapse,
# stay window and boarding time columns) and merge triage in.
cohort_df = load_dataset("ADS599-Capstone/raw_data", name="cohort_full", split="cohort_full").to_pandas()

cohort_with_triage = cohort_df.merge(df_triage, on=['ed_stay_id', 'subject_id'], how='inner')
print(f"cohort rows: {len(cohort_df):,}")
print(f"triage rows: {len(df_triage):,}")
print(f"merged rows: {len(cohort_with_triage):,}")
print(f"unique ed_stay_id: {cohort_with_triage['ed_stay_id'].nunique():,}")
cohort_with_triage.head()

In [ ]:
WITH_TRIAGE_DESCRIPTION = (
    "Primary cohort table with ED triage data merged in. One row per ED visit. "
    "Built by merging the processed cohort (AGAINST ADVICE removed, consecutive ED visits "
    "collapsed, stay_window and ed_boarding_time_hrs added) with ed.triage on ed_stay_id + subject_id. "
    "Triage rows for the second visit in a consecutive pair are dropped by the inner join. "
    "Intended as the main feature engineering input for the RL model."
)

ds_with_triage = Dataset.from_pandas(cohort_with_triage, split='with_triage')
ds_with_triage.push_to_hub("ADS599-Capstone/raw_data", config_name="with_triage", data_dir="cohort")
push_dataset_card("ADS599-Capstone/raw_data", config_name="with_triage", split="with_triage", data_dir="cohort", description=WITH_TRIAGE_DESCRIPTION)
print("Pushed with_triage to HuggingFace Hub.")